In [1]:
import sys
sys.path.append("../..")

In [2]:
from datasets import load_dataset
from syntax_tokenizer import SyntaxTokenizer
from model import ModelConfig, LlamaModel
from train import TrainerConfig, SimpleDataLoader, Trainer

In [3]:
ds = load_dataset('amang1802/wildeweb_cls_1M')['train']

In [4]:
# tokenizer = SyntaxTokenizer()
# tokenizer.train(ds.shuffle(seed=42).select(range(100_000))["text"])
tokenizer = SyntaxTokenizer.load("./wildeweb/tokenizer.data")

In [5]:
#tokenizer.save("./wildeweb/tokenizer.data")

In [6]:
tokenizer.vocab_size

24634

In [7]:
model_config = ModelConfig(
    is_causal=False,
    vocab_size=tokenizer.vocab_size,
    d_model=576,
    d_head=64,
    d_mlp_proj=1536,
    n_layers=30,
    n_kv_heads=3,
    n_attn_heads=9,
    rms_norm_eps=1e-5,
    initializer_range=0.02,
    rope_theta=100000.0,
    padding_idx=tokenizer.data.pad_token_id
)

In [8]:
train_config = TrainerConfig(
    is_causal=False,
    mask_ratio=0.15,
    per_device_train_batch_size=32,
    max_seq_len=512,
    num_epochs=4,
    eval_interval_steps=25,
    learning_rate=4e-3,
    grad_clip_norm=1.0,
    val_size=0.05,
    log_dir="runs/shakespeare_bidirectional",
    warmup_ratio=0.1
)

In [9]:
dataloader = SimpleDataLoader(train_config, tokenizer, texts=ds.shuffle(seed=42).select(range(100_000))["text"])

100000it [16:49, 99.11it/s]


Total tokens                   | 117,777,835


In [10]:
model = LlamaModel(model_config)
trainer = Trainer(train_config, model, tokenizer)

4it [00:00,  5.37it/s]

Num Trainable Params           | 134,581,824
Train device                   | cuda, NVIDIA GeForce RTX 3090, N=1
Training precision             | torch.bfloat16
Flash Attention                | True
torch.compile()                | True
DistributedDataParallel        | False
Batch size                     | 2,457




In [11]:
trainer.train(dataloader)

Training steps                 | 27,372 
Step: 0, Training Loss: 10.14879, LR: 0.0002000, Tokens/sec: 574.14
Step: 1, Training Loss: 8.53569, LR: 0.0002014, Tokens/sec: 729.62
Step: 2, Training Loss: 7.77846, LR: 0.0002028, Tokens/sec: 101813.51
Step: 3, Training Loss: 8.35756, LR: 0.0002042, Tokens/sec: 97227.10
Computing Eval loss, steps: 361
Step: 3, Eval Loss: 7.43086
Step: 4, Training Loss: 8.17643, LR: 0.0002056, Tokens/sec: 90122.19
Step: 5, Training Loss: 7.23003, LR: 0.0002069, Tokens/sec: 86777.11
Step: 6, Training Loss: 6.60451, LR: 0.0002083, Tokens/sec: 99228.05
Step: 7, Training Loss: 7.03246, LR: 0.0002097, Tokens/sec: 95799.99
Step: 8, Training Loss: 6.77120, LR: 0.0002111, Tokens/sec: 93277.16
Step: 9, Training Loss: 7.25002, LR: 0.0002125, Tokens/sec: 85227.83
Step: 10, Training Loss: 6.70528, LR: 0.0002139, Tokens/sec: 97644.86
Step: 11, Training Loss: 7.00243, LR: 0.0002153, Tokens/sec: 91346.95
Step: 12, Training Loss: 6.46066, LR: 0.0002167, Tokens/sec: 88826.22
S

In [12]:
trainer.save_checkpoint("wildeweb")

Saving checkpoint: wildeweb/model.checkpoint.2025-04-22--17-50-31.pt
Checkpoint saved
